# Day3_06 · 위치 데이터와 지도

울산 기상 격자점의 위도·경도를 표시하고, 실제 구·군 경계와 연결합니다.

## 이 노트북에서 확인할 내용

- 위도·경도 열을 지도 위의 점으로 표시할 수 있습니다.
- 울산 구·군 경계를 읽고 그릴 수 있습니다.
- 기상 격자점이 어느 구·군 안에 있는지 확인할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 실행하고 바로 아래 결과를 확인합니다. 값을 만드는 셀 다음에는 그 값을 화면에 표시하는 셀이 이어집니다. 수정 안내가 있는 셀에서는 주석 아래의 값만 바꿉니다.

### D3B-06-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-06-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-06-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-06-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-06-05 · 표 도구 불러오기

In [ ]:
import pandas as pd

### D3B-06-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt

### D3B-06-07 · JSON 도구 불러오기

In [ ]:
import json

### D3B-06-08 · 점이 경계 안에 있는지 확인할 도구 불러오기

In [ ]:
from matplotlib.path import Path as MapPath

### D3B-06-09 · 울산 기상 격자점 CSV 읽기

In [ ]:
weather_points = pd.read_csv(DATA_DIR / 'ulsan_weather_points.csv')

In [ ]:
weather_points

### D3B-06-10 · 위치 열 확인

In [ ]:
weather_points[['point_name', 'district', 'longitude', 'latitude']]

### D3B-06-11 · 위도와 경도로 점 표시

In [ ]:
weather_points.plot.scatter(x='longitude', y='latitude')
plt.title('Ulsan weather grid points')
plt.show()

### D3B-06-12 · 울산 구·군 경계 파일 읽기

In [ ]:
ulsan_geo = json.loads((DATA_DIR / 'ulsan_districts.geojson').read_text(encoding='utf-8'))

In [ ]:
ulsan_geo

### D3B-06-13 · 구·군 개수 확인

In [ ]:
len(ulsan_geo['features'])

### D3B-06-14 · 구·군 이름 확인

In [ ]:
[feature['properties']['district'] for feature in ulsan_geo['features']]

### D3B-06-15 · 경계 좌표를 꺼내는 함수 준비

In [ ]:
def geometry_rings(geometry):
    if geometry['type'] == 'Polygon':
        return [geometry['coordinates'][0]]
    return [polygon[0] for polygon in geometry['coordinates']]

### D3B-06-16 · 첫 구·군의 경계 좌표 확인

In [ ]:
geometry_rings(ulsan_geo['features'][0]['geometry'])[0][:5]

### D3B-06-17 · 울산 구·군 경계 그리기

In [ ]:
for feature in ulsan_geo['features']:
    for boundary in geometry_rings(feature['geometry']):
        x, y = zip(*boundary)
        plt.fill(x, y, alpha=0.15, edgecolor='black')
plt.title('Ulsan district boundaries')
plt.show()

### D3B-06-18 · 경계 위에 기상 격자점 표시

In [ ]:
for feature in ulsan_geo['features']:
    for boundary in geometry_rings(feature['geometry']):
        x, y = zip(*boundary)
        plt.fill(x, y, alpha=0.12, edgecolor='black')
plt.scatter(weather_points['longitude'], weather_points['latitude'], color='navy')
plt.title('Ulsan districts and weather grid points')
plt.show()

### D3B-06-19 · 좌표가 들어 있는 지역 찾는 함수 준비

In [ ]:
def find_district(longitude, latitude):
    for feature in ulsan_geo['features']:
        for boundary in geometry_rings(feature['geometry']):
            if MapPath(boundary).contains_point((longitude, latitude)):
                return feature['properties']['district']
    return 'Not found'

### D3B-06-20 · 기상 격자점마다 지도에서 찾은 구·군 붙이기

In [ ]:
weather_points['district_from_map'] = weather_points.apply(
    lambda row: find_district(row['longitude'], row['latitude']),
    axis=1,
)

In [ ]:
weather_points[['point_name', 'district', 'district_from_map']]

### D3B-06-21 · 기존 구·군 이름과 지도 결과 비교

In [ ]:
district_match = weather_points['district'] == weather_points['district_from_map']

In [ ]:
district_match.value_counts()

### D3B-06-22 · 연평균기온에 따라 점 색 바꾸기

In [ ]:
for feature in ulsan_geo['features']:
    for boundary in geometry_rings(feature['geometry']):
        x, y = zip(*boundary)
        plt.fill(x, y, alpha=0.08, edgecolor='black')
plt.scatter(weather_points['longitude'], weather_points['latitude'], c=weather_points['temperature_mean_c'], cmap='coolwarm', s=100)
plt.colorbar(label='Mean temperature (C)')
plt.title('2024 mean temperature at Ulsan grid points')
plt.show()

## 반복 실습 · 위치와 기상값을 함께 읽기

다섯 격자점의 동서남북 위치를 찾고, 구·군과 연간 강수량을 지도에서 다시 확인합니다.

### D3B-06-23 · 기상 격자점 표의 크기 확인

In [ ]:
weather_points.shape

### D3B-06-24 · 기상 격자점 표의 열 이름 확인

In [ ]:
weather_points.columns

### D3B-06-25 · 가장 북쪽 격자점 확인

In [ ]:
weather_points.nlargest(1, 'latitude')[['point_name', 'latitude']]

### D3B-06-26 · 가장 남쪽 격자점 확인

In [ ]:
weather_points.nsmallest(1, 'latitude')[['point_name', 'latitude']]

### D3B-06-27 · 가장 동쪽 격자점 확인

In [ ]:
weather_points.nlargest(1, 'longitude')[['point_name', 'longitude']]

### D3B-06-28 · 가장 서쪽 격자점 확인

In [ ]:
weather_points.nsmallest(1, 'longitude')[['point_name', 'longitude']]

### D3B-06-29 · 지도에서 강조할 구·군 정하기

In [ ]:
# 남구를 북구, 중구, 동구, 울주군 중 하나로 바꿔 보세요.
highlight_district = '남구'

In [ ]:
highlight_district

### D3B-06-30 · 강조할 구·군의 격자점 찾기

In [ ]:
highlight_row = weather_points[weather_points['district'] == highlight_district]

In [ ]:
highlight_row

### D3B-06-31 · 선택한 격자점을 붉은 점으로 강조

In [ ]:
for feature in ulsan_geo['features']:
    for boundary in geometry_rings(feature['geometry']):
        x, y = zip(*boundary)
        plt.fill(x, y, alpha=0.08, edgecolor='black')
plt.scatter(weather_points['longitude'], weather_points['latitude'], color='lightgray')
plt.scatter(highlight_row['longitude'], highlight_row['latitude'], color='red', s=130)
plt.title('Selected Ulsan district')
plt.show()

### D3B-06-32 · 지도 연결에 실패한 격자점 확인

In [ ]:
weather_points[~district_match][['point_name', 'longitude', 'latitude']]

### D3B-06-33 · 연간 강수량 기준 정하기

In [ ]:
# 1400을 1300 또는 1500으로 바꿔 보세요.
precipitation_limit = 1400

In [ ]:
precipitation_limit

### D3B-06-34 · 기준보다 강수량이 많은 격자점 찾기

In [ ]:
wet_points = weather_points[weather_points['precipitation_total_mm'] >= precipitation_limit]

In [ ]:
wet_points[['district', 'precipitation_total_mm']]

### D3B-06-35 · 연간 강수량에 따라 점 색 바꾸기

In [ ]:
for feature in ulsan_geo['features']:
    for boundary in geometry_rings(feature['geometry']):
        x, y = zip(*boundary)
        plt.fill(x, y, alpha=0.08, edgecolor='black')
plt.scatter(weather_points['longitude'], weather_points['latitude'], c=weather_points['precipitation_total_mm'], cmap='Blues', s=110)
plt.colorbar(label='Annual precipitation (mm)')
plt.title('2024 precipitation at Ulsan grid points')
plt.show()

### D3B-06-36 · 연평균기온이 높은 순서로 정렬

In [ ]:
temperature_order = weather_points.sort_values('temperature_mean_c', ascending=False)

In [ ]:
temperature_order[['district', 'temperature_mean_c']]

## 정리

위도·경도 점이 어느 구·군 경계 안에 있는지 확인해 지역 이름을 붙였습니다. 전문가들은 이런 작업을 **공간 결합**이라고 부릅니다.